## **Neural Network setup**

In [ ]:
import torch
import torch.nn as nn

class ResidualBModel(nn.Module):
    """
    Correction model for residual dynamics learning.

    Input features: [sin/cos(roll,pitch,yaw), v(3), w(3), u(4)]
    Output: delta_x_dot in physical derivative units (B,12).
    """
    def __init__(self, hidden_layers_size, activation_fn, T_inv=None, output_activation=nn.Identity):
        super(ResidualBModel, self).__init__()
        self.n_state = 12
        n_control = 4
        n_input = 6 + 3 + 3 + n_control  # trig(angles) + linear vel + angular vel + controls = 16

        layers = [nn.Linear(n_input, hidden_layers_size[0]), activation_fn()]
        for i in range(len(hidden_layers_size) - 1):
            layers.append(nn.Linear(hidden_layers_size[i], hidden_layers_size[i + 1]))
            layers.append(activation_fn())

        # Final linear layer + explicit output activation (Identity by default).
        layers.append(nn.Linear(hidden_layers_size[-1], self.n_state))
        layers.append(output_activation())
        self.corr_net = nn.Sequential(*layers)

        for m in self.corr_net.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

        # Optional diagonal scaling for correction output.
        if T_inv is None:
            t_inv = torch.ones(self.n_state, dtype=torch.float32)
        else:
            t_inv = torch.as_tensor(T_inv, dtype=torch.float32).view(-1)
            if t_inv.numel() != self.n_state:
                raise ValueError(f"T_inv must have {self.n_state} elements, got {t_inv.numel()}")
        self.register_buffer("T_inv", t_inv)

    @staticmethod
    def build_features(state_vector, control_input):
        """
        state_vector: (B,12) [x,y,z, roll,pitch,yaw, vx,vy,vz, w_roll,w_pitch,w_yaw]
        control_input: (B,4)
        returns z: (B,16)
        """
        roll = state_vector[:, 3]
        pitch = state_vector[:, 4]
        yaw = state_vector[:, 5]

        trig = torch.stack([
            torch.sin(roll), torch.cos(roll),
            torch.sin(pitch), torch.cos(pitch),
            torch.sin(yaw), torch.cos(yaw),
        ], dim=1)  # (B,6)

        v = state_vector[:, 6:9]     # (B,3)
        w = state_vector[:, 9:12]    # (B,3)

        z = torch.cat([trig, v, w, control_input], dim=1)  # (B,16)
        return z

    def forward(self, state_vector, control_input):
        z = self.build_features(state_vector, control_input)
        corr_norm = self.corr_net(z)
        corr = corr_norm * self.T_inv
        return corr

## **Data-loss function**

In [ ]:
def diff_sysdyn_dataset(state_vector, control_input, mass, inertia, g, dt, X_next):
    """
    Compute the error between the approximated system dynamics and the dataset derivatives.
    """
    # Compute the approximated derivatives using the system dynamics function
    state_vector_dot_approx = system_dynamics(state_vector, control_input, mass, inertia, g)

    # Euler integration to get the next state from the current state and the approximated derivatives
    state_vector_next_approx = state_vector + state_vector_dot_approx * dt

    # Compute the error between the approximated and actual next states
    delta_error = X_next - state_vector_next_approx

    return delta_error, state_vector_next_approx

In [ ]:
import torch
import torch.nn as nn
# from a_System_dynamics.system_dynamics import system_dynamics

_mse_loss = nn.MSELoss()


def data_loss(model, X_curr, U_curr_NN, X_next, dt,
              mass=2.0, inertia=torch.tensor([0.0217, 0.0217, 0.04]), g=9.81,
              channel_weights=None, lambda_corr=0):
    if dt.ndim == 1:
        dt = dt.view(-1, 1)

    # baseline physics derivative
    x_dot_phys = system_dynamics(X_curr, U_curr_NN, mass, inertia=inertia, g=g)  # (B,12)

    ### Do we take System Dynamics state derivative as input to the NN, or do we feed the NN with the current state and control?

    # NN correction in derivative space (model builds its own features internally)
    delta_x_dot = model(X_curr, U_curr_NN)  # (B,12)

    delta_x_pred = delta_x_dot * dt  # Correction in state space after integration

    delta_error, _ = diff_sysdyn_dataset(X_curr, U_curr_NN, mass, inertia, g, dt, X_next)

    # channel-wise MSE, real error - approx error (to keep your weighting approach)
    per_channel_losses = torch.mean((delta_x_pred - delta_error) ** 2, dim=0)  # (12,)

    if channel_weights is None:
        cw = torch.ones(12, device=X_curr.device, dtype=X_curr.dtype)
    else:
        cw = channel_weights.to(device=X_curr.device, dtype=X_curr.dtype)

    loss_data_corr = torch.sum(cw * per_channel_losses)

    # Optional: discourage huge corrections (PINN-ish regularization)
    loss_corr = torch.mean(delta_x_dot ** 2)

    total_loss = loss_data_corr + lambda_corr * loss_corr
    return total_loss, {"loss_data_corr": loss_data_corr.detach(), "loss_corr": loss_corr.detach()}